# GigaPath Attention Heatmap (RunPod)

RunPod 원격 GPU 환경에서 `/workspace/data/raw/*.svs`를 타일링 → GigaPath 추론 → 마지막 self-attention(CLS→토큰)으로 타일 점수 → 슬라이드 썸네일에 히트맵을 오버레이하는 전 과정을 담았습니다.

아래 셀을 순서대로 실행하면 `/workspace/data/work/heatmap/<슬라이드>_heatmap.png`가 생성되고, 마지막 셀로 로컬로 가져올 수 있습니다.

## 0. 준비사항
- RunPod 인스턴스가 켜져 있고, SSH 정보가 `gigapath_runpod_modeling.ipynb`와 동일하게 유효해야 합니다.
- RunPod 내부에 CUDA가 켜져 있어야 합니다(`nvidia-smi`로 확인).
- GigaPath 모델은 HuggingFace gated 모델이므로 토큰이 필요합니다. 토큰을 RunPod 환경변수(`HF_TOKEN` 혹은 `HUGGINGFACE_HUB_TOKEN`)에 넣어두세요.
- 패키지: `openslide-python`, `timm`, `huggingface_hub`, `scikit-image`, `matplotlib`가 원격에 설치되어 있어야 합니다. 필요하면 `pip install`을 추가하세요.

In [ ]:
import os, shlex, subprocess, json, textwrap
from pathlib import Path

# gigapath_runpod_modeling.ipynb와 동일하게 SSH 정보를 맞춰주세요.
SSH_HOST_DIRECT = 'root@69.30.85.100'   # RunPod direct TCP
SSH_PORT_DIRECT = 22027                 # RunPod direct TCP port
SSH_KEY = '~/.ssh/runpod_peter'          # 필요없으면 None
SSH_EXTRA_OPTS = ''                      # 필요 시 '-T' 등

# 원격 경로 설정
REMOTE_BASE = '/workspace/data'
REMOTE_RAW = f"{REMOTE_BASE}/raw"
REMOTE_WORK = f"{REMOTE_BASE}/work"
REMOTE_HEATMAP = f"{REMOTE_WORK}/heatmap"

# 히트맵 파라미터
TILE_SIZE = 256
STRIDE = 256
LEVEL = 1
MIN_TISSUE_RATIO = 0.2  # 다운샘플 레벨에서 느슨하게 (너무 빡빡하면 타일 0개)
BATCH_SIZE = 16
MODEL_NAME = 'hf_hub:prov-gigapath/prov-gigapath'

# 사용할 SVS (None이면 RAW_DIR에서 첫 번째 파일 자동 선택)
REMOTE_SVS = None  # 예: '/workspace/data/raw/slide1.svs'

# .env에서 HF 토큰 로드
def _find_env_file(start: Path):
    for p in [start] + list(start.parents):
        cand = p / '.env'
        if cand.exists():
            return cand
    return None

def _load_hf_token_local():
    token = (os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN') or os.environ.get('HF_HUB_TOKEN'))
    if token:
        return token
    # ~/.ssh 경로 내 토큰 파일 시도
    for cand in ['~/.ssh/hf_token', '~/.ssh/hf_token.txt']:
        p = Path(os.path.expanduser(cand))
        if p.exists():
            val = p.read_text().strip()
            if val:
                return val
    env_path = _find_env_file(Path.cwd())
    if env_path:
        for line in env_path.read_text().splitlines():
            line = line.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue
            k, v = line.split('=', 1)
            if k.strip() in ('HF_TOKEN', 'HUGGINGFACE_HUB_TOKEN', 'HUGGINGFACE_TOKEN', 'HF_HUB_TOKEN'):
                return v.strip()
    # 최종 fallback 없음: 토큰이 없으면 실행 시 오류


HF_TOKEN_LOCAL = _load_hf_token_local()
print('REMOTE_RAW =', REMOTE_RAW)
print('MODEL_NAME =', MODEL_NAME)
print('HF_TOKEN_LOCAL:', 'set' if HF_TOKEN_LOCAL else 'missing')


## 1. SSH/rsync 헬퍼
- `run_ssh(cmd)`: 원격에서 명령 실행
- `rsync_from(remote, local)`: 원격 → 로컬 파일/디렉터리 다운로드

In [ ]:
def _ssh_base_opts():
    opts = ['-p', str(SSH_PORT_DIRECT)]
    if SSH_KEY:
        opts = ['-i', os.path.expanduser(SSH_KEY)] + opts
    if SSH_EXTRA_OPTS:
        opts += SSH_EXTRA_OPTS.split()
    return opts

def _base_ssh():
    return ['ssh'] + _ssh_base_opts() + [SSH_HOST_DIRECT]

def run_ssh(cmd: str, check=True):
    full = _base_ssh() + [cmd]
    print('[SSH]', cmd[:200])
    proc = subprocess.run(full, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if check and proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, proc.args)
    return proc

def rsync_from(remote_path: str, local_path: str, delete=False):
    ssh_cmd = 'ssh ' + ' '.join(_ssh_base_opts())
    opts = ['-avz', '-e', ssh_cmd]
    if delete:
        opts.append('--delete')
    remote = f"{SSH_HOST_DIRECT}:{remote_path}"
    cmd = ['rsync'] + opts + [remote, local_path]
    print('[rsync]', ' '.join(cmd))
    return subprocess.run(cmd, check=True)

print('SSH helper ready')


## 2. 원격 GPU/환경 확인
필수는 아니지만 CUDA 인식 여부와 openslide 설치 여부를 확인합니다.

In [ ]:
run_ssh("nvidia-smi || echo 'no gpu'")
run_ssh("python - <<'PY'\nimport torch, platform\nprint('torch', torch.__version__, 'cuda', torch.cuda.is_available())\ntry:\n import openslide\n print('openslide ok')\nexcept Exception as e:\n print('openslide missing', e)\nPY")

## 3. 처리할 슬라이드 선택
- `REMOTE_SVS`를 직접 지정하거나, None이면 RAW 디렉터리의 첫 번째 `.svs`를 고릅니다.

In [ ]:
if REMOTE_SVS is None:
    pick_cmd = f"python - <<'PY'\nfrom pathlib import Path\nraw = Path('{REMOTE_RAW}')\ncands = sorted(list(raw.glob('*.svs')) + list(raw.glob('*.tif')))\nprint(cands[0] if cands else '')\nPY"
    proc = subprocess.run(_base_ssh() + [pick_cmd], capture_output=True, text=True)
    REMOTE_SVS = proc.stdout.strip()
    if not REMOTE_SVS:
        raise SystemExit('원격 RAW에서 SVS를 찾지 못했습니다. REMOTE_SVS를 직접 지정하세요.')
print('사용할 슬라이드:', REMOTE_SVS)

## 4. 원격에서 타일링→GigaPath 추론→히트맵 생성
- 마지막 self-attention CLS→토큰(head 평균)으로 타일 점수를 계산합니다.
- 썸네일에 스케일을 맞춰 히트맵을 오버레이하고 PNG로 저장합니다.
- 필요한 패키지가 없으면 스크립트 앞부분에 `pip install`을 추가하세요.

In [ ]:
remote_script_template = """
python - <<'PY'
import os, time
from pathlib import Path
import numpy as np
import torch
import timm
import openslide
from PIL import Image
from huggingface_hub import login
from timm.data import create_transform, resolve_model_data_config
from skimage import color, filters
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
import traceback, sys

SVS_PATH = Path('{REMOTE_SVS}')
TILE_SIZE = {TILE_SIZE}
STRIDE = {STRIDE}
LEVEL = {LEVEL}
MIN_TISSUE_RATIO = {MIN_TISSUE_RATIO}
BATCH_SIZE = {BATCH_SIZE}
MODEL_NAME = '{MODEL_NAME}'
OUT_DIR = Path('{REMOTE_HEATMAP}')
OUT_DIR.mkdir(parents=True, exist_ok=True)

token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
if not token:
    raise SystemExit('HF token missing; set HF_TOKEN/HUGGINGFACE_HUB_TOKEN')
from huggingface_hub import HfApi
api = HfApi()
try:
    who = api.whoami(token=token)
    print('HF whoami:', who.get('name') or who.get('user'))
except Exception as e:
    raise SystemExit('HF token unauthorized or invalid: %s' % e)
if token:
    try:
        login(token=token)
        print('HF login ok')
    except Exception as e:
        print('HF login failed:', e)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device != 'cuda':
    raise SystemExit('CUDA device not available on RunPod')

def tissue_ratio(np_rgb):
    hsv = color.rgb2hsv(np_rgb)
    sat = hsv[:, :, 1]
    thresh = filters.threshold_otsu(sat)
    return float((sat > thresh).mean())

def iter_tiles(slide, level, tile, stride, min_ratio):
    # level 좌표계를 level-0로 환산해 읽어야 배경 오판을 줄임
    down = slide.level_downsamples[level]
    down = float(down if isinstance(down, (int, float)) else down[0])
    width, height = slide.level_dimensions[level]
    for y_lv in range(0, height, stride):
        for x_lv in range(0, width, stride):
            x0 = int(x_lv * down)
            y0 = int(y_lv * down)
            region = slide.read_region((x0, y0), level, (tile, tile)).convert('RGB')
            arr = np.asarray(region)
            if arr.shape[0] < tile or arr.shape[1] < tile:
                continue
            if tissue_ratio(arr) < min_ratio:
                continue
            yield (x0, y0), region

def make_thumbnail(slide, max_size=2048):
    w, h = slide.level_dimensions[0]
    scale = max(w, h) / max_size
    thumb = slide.get_thumbnail((int(w/scale), int(h/scale))).convert('RGB')
    return thumb, scale

def forward_last_attn(vit_model, batch_imgs):
    with torch.inference_mode():
        x = vit_model.patch_embed(batch_imgs)
        B, N, C = x.shape
        if getattr(vit_model, 'cls_token', None) is not None:
            cls_tokens = vit_model.cls_token.expand(B, -1, -1)
            if getattr(vit_model, 'dist_token', None) is not None and vit_model.num_prefix_tokens == 2:
                cls_tokens = torch.cat((cls_tokens, vit_model.dist_token.expand(B, -1, -1)), dim=1)
            x = torch.cat((cls_tokens, x), dim=1)
        if getattr(vit_model, 'pos_embed', None) is not None:
            x = x + vit_model.pos_embed
        x = vit_model.pos_drop(x)
        for blk in vit_model.blocks[:-1]:
            x = blk(x)
        blk = vit_model.blocks[-1]
        y = blk.norm1(x)
        qkv = blk.attn.qkv(y).reshape(B, y.shape[1], 3, blk.attn.num_heads, C // blk.attn.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * blk.attn.scale
        attn = attn.softmax(dim=-1)
        attn = blk.attn.attn_drop(attn)
        x_attn = (attn @ v).transpose(1, 2).reshape(B, y.shape[1], C)
        x_attn = blk.attn.proj(x_attn)
        x_attn = blk.attn.proj_drop(x_attn)
        x = x + blk.drop_path1(x_attn)
        x = x + blk.drop_path2(blk.mlp(blk.norm2(x)))
        x = vit_model.norm(x)
    attn_mean = attn.mean(dim=1)
    return x, attn_mean

try:
    slide = openslide.OpenSlide(str(SVS_PATH))
    thumb, scale = make_thumbnail(slide, max_size=2048)
    down = slide.level_downsamples[LEVEL]
    down = float(down if isinstance(down, (int, float)) else down[0])
    tile_size_lv0 = int(TILE_SIZE * down)
    down = slide.level_downsamples[LEVEL]
    down = float(down if isinstance(down, (int, float)) else down[0])
    w0, h0 = slide.level_dimensions[0]
    print('level-0 size', (w0, h0), 'thumb', thumb.size, 'scale', scale, 'down', down)

    tiles = list(iter_tiles(slide, LEVEL, TILE_SIZE, STRIDE, MIN_TISSUE_RATIO))
    print('tiles:', len(tiles))
    if not tiles:
        raise SystemExit('조직 타일 없음. 파라미터를 조정하세요')

    model = timm.create_model(MODEL_NAME, pretrained=True).to(device)
    model.eval()
    cfg = resolve_model_data_config(model)
    transform = create_transform(**cfg, is_training=False)

    coords, scores = [], []
    start = time.time()

    def chunk(seq, n):
        for i in range(0, len(seq), n):
            yield seq[i:i+n]

    for bi, batch in enumerate(chunk(tiles, BATCH_SIZE)):
        imgs = [transform(img).unsqueeze(0) for (_, img) in batch]
        bt = torch.cat(imgs, dim=0).to(device, non_blocking=True)
        tokens, attn = forward_last_attn(model, bt)
        cls_row = attn[:, 0, 1:]
        tile_score = cls_row.mean(dim=1).detach().cpu().numpy()
        coords.extend([xy for (xy, _) in batch])
        scores.extend(tile_score.tolist())
        if (bi+1) % 10 == 0:
            print('batch %d, tiles %d, elapsed %.1fm' % (bi+1, len(coords), (time.time()-start)/60))

    coords = np.array(coords, dtype=np.int32)
    scores = np.array(scores, dtype=np.float32)
    print('score range', float(scores.min()), float(scores.max()))

    thumb_np = np.array(thumb)
    heat = np.zeros((thumb_np.shape[0], thumb_np.shape[1]), dtype=np.float32)
    cnt = np.zeros_like(heat)
    for (x, y), s in zip(coords, scores):
        # coords는 level-0 좌표
        x0 = int(x / scale)
        y0 = int(y / scale)
        x1 = int((x + tile_size_lv0) / scale)
        y1 = int((y + tile_size_lv0) / scale)
        heat[y0:y1, x0:x1] += s
        cnt[y0:y1, x0:x1] += 1
        heat[y0:y1, x0:x1] += s
        cnt[y0:y1, x0:x1] += 1

    mask = cnt > 0
    heat[mask] /= cnt[mask]
    heat = gaussian_filter(heat, sigma=8)
    if mask.any():
        p2, p98 = np.percentile(heat[mask], [2, 98])
    else:
        p2, p98 = heat.min(), heat.max()
    heat = np.clip((heat - p2) / (p98 - p2 + 1e-6), 0, 1)
    cmap = plt.get_cmap('turbo')
    heat_rgb = (cmap(heat)[:, :, :3] * 255).astype(np.uint8)
    alpha = 0.45
    overlay = (alpha * heat_rgb + (1 - alpha) * thumb_np).astype(np.uint8)

    out_path = OUT_DIR / ('%s_heatmap.png' % SVS_PATH.stem)
    Image.fromarray(overlay).save(out_path)
    print('saved overlay ->', out_path)
    print('total time %.1fm' % ((time.time()-start)/60))
except Exception:
    traceback.print_exc()
    sys.exit(1)
PY
"""

remote_script = remote_script_template.format(
    REMOTE_SVS=REMOTE_SVS,
    TILE_SIZE=TILE_SIZE,
    STRIDE=STRIDE,
    LEVEL=LEVEL,
    MIN_TISSUE_RATIO=MIN_TISSUE_RATIO,
    BATCH_SIZE=BATCH_SIZE,
    MODEL_NAME=MODEL_NAME,
    REMOTE_HEATMAP=REMOTE_HEATMAP,
)

env_prefix = ''
if HF_TOKEN_LOCAL:
    env_prefix = f"env HF_TOKEN={HF_TOKEN_LOCAL} HUGGINGFACE_HUB_TOKEN={HF_TOKEN_LOCAL} HUGGINGFACE_TOKEN={HF_TOKEN_LOCAL} HF_HUB_TOKEN={HF_TOKEN_LOCAL} "
cmd = env_prefix + remote_script.lstrip()
proc = run_ssh(cmd, check=False)
print('remote return code:', proc.returncode)
if proc.returncode != 0:
    print('원격 실행 실패. 위 stderr/traceback을 확인하세요.')


## 5. 생성된 히트맵 로컬로 받기 (옵션)
- 원격에 저장된 PNG를 로컬 `PoC/v1/output/` 아래로 가져옵니다.

In [ ]:
local_out = Path('PoC/v1/output')
local_out.mkdir(parents=True, exist_ok=True)
rsync_from(f"{REMOTE_HEATMAP}/", str(local_out) + '/')
print('다운로드 완료 →', local_out)
list(local_out.glob('*.png'))[:5]

## 6. 파라미터 튜닝 팁
- `TILE_SIZE/STRIDE`: 작게 하면 해상도↑, 속도↓. 크게 하면 빠르지만 블러 증가.
- `MIN_TISSUE_RATIO`: 배경 필터. 너무 높으면 타일이 사라질 수 있으니 0.3~0.6 사이로 조정.
- `sigma`(가우시안)나 `alpha`(오버레이 투명도)는 코드에서 직접 수정 가능합니다.
- 멀티스케일 필요 시 `LEVEL`을 1로 내리고 `TILE_SIZE`를 224로 유지하거나 배율에 맞게 변경하세요.